In [7]:
# Import some libraries and especially load environment variables
from langchain_openai  import AzureChatOpenAI
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv()) # read local .env file
# set an environment varibale
os.environ["LANGCHAIN_PROJECT"] = "01_basic"

print(os.getenv("AZURE_ENDPOINT"))

https://alkopenai2.openai.azure.com/


In [8]:
llm = AzureChatOpenAI(
	temperature=0,
    openai_api_version="2023-09-01-preview",
    deployment_name="GPT-4o-mini", #Deployment name
    azure_endpoint=os.environ["OPENAI_API_BASE"],
    model_name="GPT-4o-mini"
)

In [9]:
# I the import my custom function to work with audio and videp
from plugins.CallJarvis import Tasks
tasks = Tasks()

In [10]:

# you can test the function
result = tasks.change_task_title("Task_1", "new title")
# print(f"New version of the task: {result}")

Changing title of task Task_1 to new title


In [11]:
# Then create tools list
from langchain.tools import  StructuredTool

tools = [
    StructuredTool.from_function(
        func=tasks.change_task_title,
        name="ChangeTaskTitle",
        description="Change the title of a task.",
        args_schema=tasks.ChangeTaskTitleInput
    ),
    StructuredTool.from_function(
        func=tasks.search_task,
        name="SearchTask",
        description="Can search task with a full Text string.",
        args_schema=tasks.SearchTaskInput
    ),
    StructuredTool.from_function(
        func=tasks.load_task,
        name="LoadTask",
        description="Given a task_id can load task detail in json format.",
        args_schema=tasks.LoadTaskInput
    )
]

In [12]:
from langchain.agents import create_tool_calling_agent
from langchain import hub

# Get the prompt to use - you can modify this!
prompt = hub.pull("hwchase17/openai-functions-agent")
agent = create_tool_calling_agent(tools=tools, llm=llm, prompt=prompt)

In [ ]:
# Everthing is ready to start the conversation
from langchain.agents import AgentExecutor
from pprint import pprint   

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
result =agent_executor.invoke({"input": """
Search all task containing "test123" and if the due date is in year 2021 add the word "BOMB" as prefix of the title
"""})


ValidationError: 1 validation error for AgentExecutor
tools
  Field required [type=missing, input_value={'agent': RunnableMultiAc...=True), 'verbose': True}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.9/v/missing

In [ ]:

pprint(result)